In [2]:
import pandas as pd
import numpy as np

In [3]:
print(pd.__version__)

3.0.6


In [4]:
df_train1 = pd.read_csv(
    "C:/Users/goura/Sem5/Machine Learning/BT2024213/BT2024213/BT2024213_train_var2.csv"
)
display(df_train1.head());

,x1,x2,x3,y
0,-1.000000,0.338439,-0.246107,-2.316238
1,-0.298939,-0.144389,-0.361242,0.062753
2,1.000000,-1.000000,0.581159,-6.263346
3,-0.210720,0.692389,0.813495,2.802039
4,0.221644,0.880271,-1.000000,-0.331873


In [5]:
df_test1 = pd.read_csv(
    "C:/Users/goura/Sem5/Machine Learning/BT2024213/BT2024213/BT2024213_test_var2.csv"
)
display(df_test1.head());

,x1,x2,x3
0,0.398138,-0.765946,0.201898
1,1.000000,-0.229396,-1.000000
2,0.658104,0.303319,1.000000
3,-0.477849,-1.000000,0.043375
4,-0.174990,0.969894,0.107388


In [6]:
Y = df_train1.iloc[:, 3:].values
print(Y.shape)
print(Y[0])

(1000, 1)
[-2.31623802]


In [7]:
N = df_train1.shape[0]
ones = np.ones((N,1))
X_raw = df_train1.iloc[:, :3].values
X = np.hstack((ones, X_raw));
print(X[0])
print(X.shape)

[ 1.         -1.          0.33843935 -0.24610721]
(1000, 4)


In [8]:
Y_raw = df_train1.iloc[:, -1].values

In [9]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA compiled into PyTorch:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())



PyTorch: 2.11.0+cu128
CUDA compiled into PyTorch: 12.8
CUDA available: True


In [10]:
import torch
import torch.optim as optim
import numpy as np
import time
import gc
from itertools import combinations_with_replacement

# Ensure we use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def get_polynomial_features_torch_with_interactions(X_raw, degree):
    N, D = X_raw.shape
    # Start with bias (ones)
    features = [torch.ones((N, 1), dtype=torch.float64, device=device)]

    # Generate all combinations of features from degree 1 up to 'degree'
    for d in range(1, degree + 1):
        for combos in combinations_with_replacement(range(D), d):
            # Compute product of selected features for this combination
            prod = torch.prod(X_raw[:, combos], dim=1, keepdim=True)
            features.append(prod)

    return torch.hstack(features)

X_raw_torch = torch.tensor(X_raw, dtype=torch.float64, device=device)
Y_torch = torch.tensor(Y_raw, dtype=torch.float64, device=device).reshape(-1, 1)

test_feat = get_polynomial_features_torch_with_interactions(X_raw_torch, 3)
print(f"New feature set shape for degree 3 (including interactions): {test_feat.shape}")

Using device: cuda
New feature set shape for degree 3 (including interactions): torch.Size([1000, 20])


In [11]:
print("Device:", device)
print("X_raw:", X_raw_torch.device)


Device: cuda
X_raw: cuda:0


In [12]:
def elastic_net_adam_torch(X, Y, l1_ratio=0.5, alpha=0.01, lr=0.05, epochs=2000):
    N, D = X.shape
    beta = torch.zeros((D, 1), dtype=torch.float64, device=X.device, requires_grad=True)
    optimizer = torch.optim.Adam([beta], lr=lr)

    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = X @ beta
        mse = torch.mean((pred - Y) ** 2)
        # Penalty excludes the bias term (first column of ones)
        l1 = torch.sum(torch.abs(beta[1:]))
        l2 = torch.sum(beta[1:] ** 2)
        loss = mse + alpha * (l1_ratio * l1 + 0.5 * (1 - l1_ratio) * l2)
        loss.backward()
        optimizer.step()

    return beta.detach()

In [ ]:
# Rerun grid search using new comprehensive interaction feature sets
alphas = [0.005, 0.01, 0.05]
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
degrees = [2, 3, 4, 5, 6,7] # Higher degrees with interactions will produce large numbers of features
K_folds = 10

best_mse = float('inf')
best_params = {}

start_time = time.time()
print("Starting parallelized PyTorch autograd grid search with interactions...\n")

for deg in degrees:
    X_poly = get_polynomial_features_torch_with_interactions(X_raw_torch, deg)

    # Split setup
    torch.manual_seed(42)
    indices = torch.randperm(X_poly.size(0), device=device)
    X_poly_shuffled = X_poly[indices]
    Y_shuffled = Y_torch[indices]

    X_folds = torch.chunk(X_poly_shuffled, K_folds, dim=0)
    Y_folds = torch.chunk(Y_shuffled, K_folds, dim=0)

    for alpha in alphas:
        for l1_ratio in l1_ratios:
            validation_mses = []

            for i in range(K_folds):
                X_val, Y_val = X_folds[i], Y_folds[i]

                X_train = torch.cat([X_folds[j] for j in range(K_folds) if j != i], dim=0)
                Y_train = torch.cat([Y_folds[j] for j in range(K_folds) if j != i], dim=0)

                beta_en = elastic_net_adam_torch(
                    X_train, Y_train, l1_ratio=l1_ratio, alpha=alpha, lr=0.05, epochs= 2000
                )

                Y_val_pred = X_val.matmul(beta_en)
                fold_mse = torch.mean((Y_val - Y_val_pred) ** 2).item()
                validation_mses.append(fold_mse)

            mean_mse = np.mean(validation_mses)
            if mean_mse < best_mse:
                best_mse = mean_mse
                best_params = {'degree': deg, 'alpha': alpha, 'l1_ratio': l1_ratio}
                print(f"* New Best! Degree: {deg}, Alpha: {alpha}, L1 Ratio: {l1_ratio} -> Mean CV MSE: {mean_mse:.5f}")

    del X_poly
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

end_time = time.time()
print(f"\nGrid Search completed in {end_time - start_time:.2f} seconds!")
print(f"Optimal Parameters: {best_params}")
print(f"Lowest Validation MSE achieved: {best_mse:.5f}")

Starting parallelized PyTorch autograd grid search with interactions...

* New Best! Degree: 2, Alpha: 0.005, L1 Ratio: 0.1 -> Mean CV MSE: 21.75004
* New Best! Degree: 2, Alpha: 0.01, L1 Ratio: 0.1 -> Mean CV MSE: 21.74970
* New Best! Degree: 3, Alpha: 0.005, L1 Ratio: 0.1 -> Mean CV MSE: 11.83932
* New Best! Degree: 3, Alpha: 0.005, L1 Ratio: 0.3 -> Mean CV MSE: 11.83439
* New Best! Degree: 3, Alpha: 0.005, L1 Ratio: 0.5 -> Mean CV MSE: 11.83101
* New Best! Degree: 3, Alpha: 0.005, L1 Ratio: 0.7 -> Mean CV MSE: 11.82936
* New Best! Degree: 4, Alpha: 0.005, L1 Ratio: 0.1 -> Mean CV MSE: 3.98711
* New Best! Degree: 4, Alpha: 0.005, L1 Ratio: 0.3 -> Mean CV MSE: 3.94214
* New Best! Degree: 4, Alpha: 0.005, L1 Ratio: 0.5 -> Mean CV MSE: 3.89959
* New Best! Degree: 4, Alpha: 0.005, L1 Ratio: 0.7 -> Mean CV MSE: 3.86090
* New Best! Degree: 4, Alpha: 0.005, L1 Ratio: 0.9 -> Mean CV MSE: 3.82772
* New Best! Degree: 5, Alpha: 0.005, L1 Ratio: 0.1 -> Mean CV MSE: 2.24169
* New Best! Degree: 5,

In [13]:
y = Y_torch.cpu().numpy().ravel()
baseline_mse = np.var(y)
print(f"Baseline MSE (predict mean): {baseline_mse:.4f}")
print(f"R² from CV: {1 - best_mse / baseline_mse:.4f}")

Baseline MSE (predict mean): 47.6516


NameError: name 'best_mse' is not defined

In [15]:
import numpy as np
import pandas as pd
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures, StandardScaler

X = X_raw_torch.cpu().numpy()
y = Y_torch.cpu().numpy().ravel()
kf = KFold(n_splits=10, shuffle=True, random_state=2)

rows = []
for deg in range(2, 20):
    pipe = make_pipeline(
        MinMaxScaler(feature_range=(-1, 1)),            # keeps x^deg from blowing up
        PolynomialFeatures(degree=deg, include_bias=False),
        StandardScaler(),                                # equal footing for the penalty
        ElasticNetCV(l1_ratio=[0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
                     alphas=60, cv=kf, max_iter=50000, n_jobs=-1),
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        pipe.fit(X, y)

    enet = pipe[-1]
    cv_mse = enet.mse_path_.mean(axis=-1).min()   # mean over folds, best (l1, alpha)
    rows.append({'degree': deg,
                 'n_features': enet.coef_.size,
                 'nonzero': int(np.sum(enet.coef_ != 0)),
                 'alpha': enet.alpha_, 'l1_ratio': enet.l1_ratio_,
                 'cv_mse': cv_mse})
    print(f"Degree {deg:>2}: CV MSE {cv_mse:.4f} | alpha {enet.alpha_:.2e} | "
          f"L1 {enet.l1_ratio_} | {rows[-1]['nonzero']}/{enet.coef_.size} nonzero")

df = pd.DataFrame(rows)
best = df.loc[df['cv_mse'].idxmin()]
print(f"\nBest degree: {int(best['degree'])}, CV MSE {best['cv_mse']:.4f}, "
      f"R² {1 - best['cv_mse'] / np.var(y):.4f}")

Degree  2: CV MSE 21.5008 | alpha 8.01e-03 | L1 0.5 | 9/9 nonzero
Degree  3: CV MSE 11.4176 | alpha 4.24e-03 | L1 0.99 | 19/19 nonzero
Degree  4: CV MSE 3.5860 | alpha 4.19e-03 | L1 1.0 | 33/34 nonzero
Degree  5: CV MSE 1.5526 | alpha 4.30e-03 | L1 1.0 | 53/55 nonzero
Degree  6: CV MSE 0.7451 | alpha 4.30e-03 | L1 1.0 | 67/83 nonzero
Degree  7: CV MSE 0.5095 | alpha 4.30e-03 | L1 1.0 | 76/119 nonzero
Degree  8: CV MSE 0.3730 | alpha 4.30e-03 | L1 1.0 | 80/164 nonzero
Degree  9: CV MSE 0.3288 | alpha 4.30e-03 | L1 1.0 | 90/219 nonzero
Degree 10: CV MSE 0.2974 | alpha 4.35e-03 | L1 0.99 | 103/285 nonzero
Degree 11: CV MSE 0.2892 | alpha 4.30e-03 | L1 1.0 | 120/363 nonzero
Degree 12: CV MSE 0.2815 | alpha 4.30e-03 | L1 1.0 | 128/454 nonzero
Degree 13: CV MSE 0.2807 | alpha 4.30e-03 | L1 1.0 | 137/559 nonzero
Degree 14: CV MSE 0.2816 | alpha 4.53e-03 | L1 0.95 | 170/679 nonzero
Degree 15: CV MSE 0.2817 | alpha 4.53e-03 | L1 0.95 | 179/815 nonzero
Degree 16: CV MSE 0.2828 | alpha 4.53e-03 |